# 🔮 Bidirectional LSTM (BiLSTM) Network
**Sequence Classification with PyTorch**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print(f'PyTorch version : {torch.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using a **Synthetic Time Series** dataset. Instead of forecasting the next value, we will frame this as a **Sequence Classification** task: determining if a given sliding window exhibits a net *Upward* or *Downward* trajectory.

In [ ]:
# Generate dataset
np.random.seed(42)
t = np.linspace(0, 100, 2000)
y = 3 * np.sin(0.1 * t) + 1.5 * np.cos(0.3 * t) + 0.02 * t + np.random.randn(2000) * 0.8
df = pd.DataFrame({'time': t, 'value': y})

print(f'Shape   : {df.shape}')
df.head()

## 3. Data Preprocessing for Classification
> We create sliding windows. For each window, the **label is 1** if the value at the end of the window is greater than the value at the start (Upward trend), and **0** otherwise (Downward/Flat).

In [ ]:
def create_sequences_with_labels(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        window = data[i:i + seq_length]
        X.append(window)
        label = 1 if window[-1] > window[0] else 0
        y.append(label)
    return np.array(X), np.array(y)

SEQ_LENGTH = 30
values = df['value'].values.reshape(-1, 1)

scaler = StandardScaler()
scaled_values = scaler.fit_transform(values).flatten()

X, y = create_sequences_with_labels(scaled_values, SEQ_LENGTH)
X = np.reshape(X, (X.shape[0], X.shape[1], 1)).astype(np.float32)
y = y.astype(np.float32)

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Class distribution (Train): {np.bincount(y_train.astype(int))}')

## 4. Build the PyTorch BiLSTM Model

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.2):
        super(BiLSTMClassifier, self).__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers, 
                              batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, 1)  # *2 because bidirectional
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.bilstm(x)
        out = out[:, -1, :]  # Take the last time step
        out = self.dropout(out)
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

model = BiLSTMClassifier(input_size=1, hidden_size=32, num_layers=1, dropout=0.2)
print(model)

## 5. Train the Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
train_losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        outputs = model(bx)
        loss = criterion(outputs, by)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            outputs = model(bx)
            preds = (outputs >= 0.5).float()
            val_correct += (preds == by).sum().item()
            val_total += bx.size(0)
    
    val_accs.append(val_correct / val_total)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Val Acc: {val_accs[-1]:.4f}')

print('Training complete!')

## 6. Evaluate on Test Set

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        outputs = model(bx)
        preds = (outputs >= 0.5).float().cpu().numpy()
        all_preds.extend(preds.flatten())
        all_targets.extend(by.numpy().flatten())

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, zero_division=0)
rec = recall_score(all_targets, all_preds, zero_division=0)
f1 = f1_score(all_targets, all_preds, zero_division=0)

print('='*50)
print('          BiLSTM Test Set Results')
print('='*50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('='*50)

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Downward (0)', 'Upward (1)'],
            yticklabels=['Downward (0)', 'Upward (1)'],
            linewidths=1, linecolor='white')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Save Model & Scaler

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/bilstm_model.pth')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved  → models/bilstm_model.pth')
print('Scaler saved → models/scaler.pkl')

## 9. Key Takeaways
> - **PyTorch Flexibility**: Building a BiLSTM in PyTorch is straightforward and highly customizable.
> - **Bidirectional Context**: By processing data in both directions, the model achieves higher accuracy on sequence classification tasks compared to unidirectional LSTMs.
> - **Deployment Friendly**: PyTorch models are lightweight and deploy easily on Streamlit Cloud without the heavy dependency resolution issues of TensorFlow on newer Python versions.